In [5]:
# # %% [markdown]
# ## setup xpu
# this is how hosuld i fetch the xpu 
# for gpu , is also sort of similar with the difference of using gpu or cuda insted of xpu 

In [6]:
import torch

device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    try:
        _ = torch.ones(1, device="xpu")
        device = torch.device("xpu")
    except RuntimeError:
        print("XPU detected but not usable, falling back to CPU.")
print("Using:", device)

Using: xpu


simple test between xpu and cpu 
NOTE : 
this is wrong , becase i need to wrap xpu anywhere i am using it 

In [ ]:
import timeit
M = torch.rand((1000,1000) , device="cpu")
%timeit M@M.T
M = torch.rand((1000,1000) , device="xpu")
%timeit M@M.T

In [8]:
import torch
import timeit

# ---------- CPU benchmark ----------
M_cpu = torch.rand((1000, 1000), device="cpu")
t_cpu = timeit.timeit(lambda: M_cpu @ M_cpu.T, number=100) / 100
print(f"CPU: {t_cpu*1000:.2f} ms per matmul")

# ---------- XPU benchmark (safe) ----------
try:
    M_xpu = torch.rand((1000, 1000), device="xpu")

    # warm-up (first call often includes kernel compilation / init)
    for _ in range(5):
        _ = M_xpu @ M_xpu.T
    torch.xpu.synchronize()

    def run():
        _ = M_xpu @ M_xpu.T
        torch.xpu.synchronize()   # wait for the GPU to actually finish

    t_xpu = timeit.timeit(run, number=100) / 100
    print(f"XPU: {t_xpu*1000:.2f} ms per matmul")
    print(f"Speedup vs CPU: {t_cpu/t_xpu:.2f}x")

except RuntimeError as e:
    print("XPU not usable, falling back to CPU only.")
    print("Reason:", e)




CPU: 7.22 ms per matmul
XPU: 1.72 ms per matmul
Speedup vs CPU: 4.20x


In [9]:
# # %% [markdown]
# autogard tset  , autogard is like i say this is variable , this is constant 

In [10]:

import torch 
x = torch.ones(5 )
y = torch.zeros(3)
w = torch.randn(5 , 3 , requires_grad=True)
b = torch.randn(3 , requires_grad=True)
z = torch.matmul(x, w ) + b 
loss = torch.nn.functional.binary_cross_entropy_with_logits(z , y)

In [11]:
import torch

# ---------- Step 1: create tensors ----------
x = torch.ones(5)                          # input, shape (5,)
y = torch.zeros(3)                         # targets, shape (3,)
w = torch.randn(5, 3, requires_grad=True)  # weights, shape (5,3)
b = torch.randn(3, requires_grad=True)     # biases,  shape (3,)

print("x:", x, "| shape:", x.shape)
print("y:", y, "| shape:", y.shape)
print("w shape:", w,"w shape:", w.shape)
print("b:", b, "| shape:", b.shape)
print()

# ---------- Step 2: linear forward ----------
xw = torch.matmul(x, w)                    # (5,) @ (5,3) = (3,)
print("x @ w    shape:", xw.shape, "→ values:", xw)
print("col sums of w:", w.sum(dim=0))      # sanity check: same as x @ w
print()

z = xw + b
print("z = x@w + b shape:", z.shape, "→ values:", z)
print()

# ---------- Step 3: loss ----------
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)
print("loss (scalar):", loss)

# manually verify (because y is all zeros):
import math
manual = sum(math.log(1 + math.exp(zi.item())) for zi in z) / 3
print("manual softplus mean:", manual)
print()

# ---------- Step 4: where are the gradients? ----------
print("w.is_leaf:", w.is_leaf, " b.is_leaf:", b.is_leaf)
print("z.is_leaf:", z.is_leaf, " loss.is_leaf:", loss.is_leaf)
print("z.grad_fn:", z.grad_fn)
print()

loss.backward()

print("w.grad shape:", w.grad.shape, "→ values:\n", w.grad)
print("b.grad:", b.grad)
print("x.grad:", x.grad)   # None — x does not require grad
print("z.grad:", z.grad)   # None — z is non-leaf

x: tensor([1., 1., 1., 1., 1.]) | shape: torch.Size([5])
y: tensor([0., 0., 0.]) | shape: torch.Size([3])
w shape: tensor([[ 0.5697,  1.2067,  0.2490],
        [-1.7971, -2.1465,  2.4192],
        [ 0.3771,  1.0850, -0.3595],
        [-0.2872, -0.1676,  0.2675],
        [ 0.1496, -0.4785,  0.0694]], requires_grad=True) w shape: torch.Size([5, 3])
b: tensor([ 0.2517, -0.0693,  0.5079], requires_grad=True) | shape: torch.Size([3])

x @ w    shape: torch.Size([3]) → values: tensor([-0.9879, -0.5008,  2.6456], grad_fn=<SqueezeBackward4>)
col sums of w: tensor([-0.9879, -0.5008,  2.6456], grad_fn=<SumBackward1>)

z = x@w + b shape: torch.Size([3]) → values: tensor([-0.7362, -0.5701,  3.1535], grad_fn=<AddBackward0>)

loss (scalar): tensor(1.3450, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>)
manual softplus mean: 1.344952166142593

w.is_leaf: True  b.is_leaf: True
z.is_leaf: False  loss.is_leaf: False
z.grad_fn: <AddBackward0 object at 0x7f8be12aa710>

w.grad shape: torch.Size([5, 3]) → 

/tmp/ipykernel_104149/2334484036.py:46: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /__w/pytorch/pytorch/build/aten/src/ATen/core/TensorBody.h:488.)
  print("z.grad:", z.grad)   # None — z is non-leaf


In [12]:
print(x.is_leaf)   
print(w.is_leaf)   
print(z.is_leaf) 
print(loss.is_leaf) 

True
True
False
False


# using torch to create modsl 
lest start using the torch actuly to do some regression 
from `low level api`  to `high level api` , start from here 


### low level api 

In [13]:
import torch 
import random as rand
import gc 
INPUT_NUMBER = 10 
DATASET_SIZE = 10
# create dataset 
X = []
Y = []
for i in range(INPUT_NUMBER):
    X.append(i*rand.randint(1,10))
    Y.append( X[i] * 5 + rand.randint(0,3))
# =++++++++++++++++ 
x_train = torch.tensor(X, dtype=torch.float32)
y_train = torch.tensor(Y, dtype=torch.float32)


lr = 0.1
n_epoch = 20  
print(f"""
information : 
lr : {lr}
epoch : {n_epoch}
input  ->  sample {X[:5]}
output ->  sample {Y[:5]}
""")

def reset():
    for name in ("w", "b", "x_train", "y_train", "x_train_mean", "x_train_std", "loss", "z"):
        if name in globals():
            del globals()[name]
    
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("state reset")


information : 
lr : 0.1
epoch : 20
input  ->  sample [0, 10, 16, 24, 20]
output ->  sample [2, 50, 83, 122, 100]



## first try , fails , becase my input was not normalize 
the loss funciton was not able to probably propagate loss becase of that 
this was my output 

Epoch 0 | loss 14723.974609375 
Epoch 1 | loss 153645072.0 
Epoch 2 | loss 1603552935936.0 
Epoch 3 | loss 1.6735857612423168e+16 
Epoch 4 | loss 1.7466771350105358e+20 
Epoch 5 | loss 1.822960716098754e+24 
Epoch 6 | loss 1.9025759331132302e+28 
Epoch 7 | loss 1.985668589270405e+32 
Epoch 8 | loss 2.0723900616446157e+36 
Epoch 9 | loss inf 
Epoch 10 | loss inf 
Epoch 11 | loss inf 
Epoch 12 | loss inf 
Epoch 13 | loss inf 
Epoch 14 | loss inf 
Epoch 15 | loss inf 
Epoch 16 | loss inf 
Epoch 17 | loss inf 
Epoch 18 | loss inf 
Epoch 19 | loss nan 

In [14]:
reset()
torch.manual_seed(1)
rand.seed(1)

w = torch.randn(1 , requires_grad=True)
b = torch.tensor(0.0 , requires_grad=True)

x_train = torch.tensor(X, dtype=torch.float32)
y_train = torch.tensor(Y, dtype=torch.float32)

for epoch in range(n_epoch) :
    z = x_train*w + b 
    # no activation functoin yet 
    loss = ((z - y_train)**2 ).mean()
    loss.backward() # type: ignore
    with torch.no_grad():
        if w.grad is not None: 
            w -= lr * w.grad        
            w.grad.zero_() # type: ignore
        if b.grad is not None: 
            b -= lr * b.grad      
            b.grad.zero_() #type: ignore 
    print(f"Epoch {epoch} | loss {loss.item()} ")

state reset
Epoch 0 | loss 19265.435546875 
Epoch 1 | loss 780924736.0 
Epoch 2 | loss 31659021828096.0 
Epoch 3 | loss 1.2834696812686213e+18 
Epoch 4 | loss 5.203240134521981e+22 
Epoch 5 | loss 2.109415437488968e+27 
Epoch 6 | loss 8.551658842882425e+31 
Epoch 7 | loss 3.466878928717831e+36 
Epoch 8 | loss inf 
Epoch 9 | loss inf 
Epoch 10 | loss inf 
Epoch 11 | loss inf 
Epoch 12 | loss inf 
Epoch 13 | loss inf 
Epoch 14 | loss inf 
Epoch 15 | loss inf 
Epoch 16 | loss inf 
Epoch 17 | loss nan 
Epoch 18 | loss nan 
Epoch 19 | loss nan 


## first try , fails , becase my input was not normalize 

In [15]:
torch.manual_seed(1)
rand.seed(1)

w = torch.randn(1 , requires_grad=True)
b = torch.tensor(0.0 , requires_grad=True)

x_train = torch.tensor(X, dtype=torch.float32)
y_train = torch.tensor(Y, dtype=torch.float32)

x_train_mean , x_train_std = x_train.mean() , x_train.std()
x_train = (x_train - x_train_mean) / x_train_std
# no test set yet 
# but base on the book , somewhere around page 77 in chapter2/feature scaling section :
# i will use the very same paramether ( x_train_mean / x_train_std ) to scale the test set 
# also no scaling on y_train yet 
#
n_epoch = 50
for epoch in range(n_epoch) :
    z = x_train*w + b 
    # no activation functoin yet 
    loss = ((z - y_train)**2 ).mean()
    loss.backward() # type: ignore
    with torch.no_grad():
        if w.grad is not None: 
            w -= lr * w.grad        
            w.grad.zero_() # type: ignore
        if b.grad is not None: 
            b -= lr * b.grad      
            b.grad.zero_() #type: ignore 
    print(f"Epoch {epoch} | loss {loss.item()} ")

Epoch 0 | loss 25409.712890625 
Epoch 1 | loss 16639.369140625 
Epoch 2 | loss 10902.9580078125 
Epoch 3 | loss 7148.68115234375 
Epoch 4 | loss 4690.15478515625 
Epoch 5 | loss 3079.18408203125 
Epoch 6 | loss 2022.9388427734375 
Epoch 7 | loss 1329.980712890625 
Epoch 8 | loss 875.0836181640625 
Epoch 9 | loss 576.281005859375 
Epoch 10 | loss 379.8913269042969 
Epoch 11 | loss 250.7346649169922 
Epoch 12 | loss 165.74362182617188 
Epoch 13 | loss 109.78163146972656 
Epoch 14 | loss 72.9118423461914 
Epoch 15 | loss 48.606658935546875 
Epoch 16 | loss 32.57474899291992 
Epoch 17 | loss 21.993831634521484 
Epoch 18 | loss 15.006582260131836 
Epoch 19 | loss 10.390002250671387 
Epoch 20 | loss 7.337885856628418 
Epoch 21 | loss 5.319056034088135 
Epoch 22 | loss 3.982980251312256 
Epoch 23 | loss 3.0982728004455566 
Epoch 24 | loss 2.512129783630371 
Epoch 25 | loss 2.1236512660980225 
Epoch 26 | loss 1.8659871816635132 
Epoch 27 | loss 1.6950359344482422 
Epoch 28 | loss 1.58157038688